## Imports

In [1]:
import argparse
import copy
import json
import random
import sys
from pathlib import Path
from typing import Optional
from types import SimpleNamespace

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

from preprocessing import preprocess_mindrove_data
from models import CNNLSTMClassifier
from task_sampler import FOMAMLTaskSampler
from fomaml_train import train_fomaml
from eval import evaluate, evaluate_full, eval_cm

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## Load in Config file & Model

In [2]:
print("GLOBAL VARIABLES")
cfg = json.load(open("fomaml_config.json"), object_hook=lambda d: SimpleNamespace(**d))
cfg.seed = random.randint(1,1000000)
for var, val in vars(cfg).items():
    print(f"{var} : {val}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nGESTURE CLASSIFIER MODEL")
model_cfg = json.load(open(cfg.model_metadata_path), object_hook=lambda d: SimpleNamespace(**d))
model = CNNLSTMClassifier(
        input_size       = model_cfg.num_raw_channels,
        num_classes      = model_cfg.num_classes,
        # conv_channels    = [64, 128],
        # kernel_size      = 7,
        # lstm_hidden_size = 128,
        # lstm_num_layers  = 2,
        # dropout          = 0.3,
        # bidirectional    = True,
    )
print(model)
print(device)

GLOBAL VARIABLES
data_dir : subject_data
model_path : artifacts/best_model_v10.pt
model_metadata_path : artifacts/metadata_v10.json
inner_lr : 0.01
inner_steps : 8
outer_lr : 0.0001
meta_epochs : 1
tasks_per_epoch : 8
k_shot : 5
q_query : 10
adapt_step : 10
adapt_lr : 0.001
seed : 108395

GESTURE CLASSIFIER MODEL
CNNLSTMClassifier(
  (cnn): Sequential(
    (0): Conv1d(8, 64, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.15, inplace=False)
    (4): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv1d(64, 128, kernel_size=(7,), stride=(1,), padding=(3,))
    (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): ReLU()
    (8): Dropout(p=0.15, inplace=False)
  )
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (head): Sequential(
    (0): Linear(

# Load in Subject Data

### Load in all data
- Training data
- Fine Tuning Subject

In [3]:
data_dir = r"./mindrove_data"
ft_data_dir = r"./fine_tune_data"

data = preprocess_mindrove_data(data_dir)
data_sub = preprocess_mindrove_data(ft_data_dir)

X = data["X"]
y = data["y"]

X_ft = data_sub["X"]
y_ft = data_sub["y"]

label_map = vars(model_cfg.raw_spec_label_map)
y = np.array([label_map[label] for label in y], dtype=np.int64)
y_ft = np.array([label_map[label] for label in y_ft], dtype=np.int64)

Number of CSV files found before exclusion: 60
Number of CSV files excluded: 0
Number of CSV files used: 60

Example file used: mindrove_data\2026-05-19T18-34-39.720895.csv

Sample rate: 500 Hz
Window size: 200 samples = 0.4 seconds
Step size: 25 samples = 0.05 seconds

raw_df shape: (1476620, 14)
Detected EMG channels: ['Channel1', 'Channel2', 'Channel3', 'Channel4', 'Channel5', 'Channel6', 'Channel7', 'Channel8']
Detected number of channels: 8

Label distribution including NaN pauses:
label
Extend     96176
Fist       92486
Flex       95566
Pro        89672
Radial     91156
Rest       96964
Sup       100258
Ulnar      97098
NaN       717244
Name: count, dtype: int64
Using new Mindrove row-level labels from the last CSV column.
NaN labels are pause/unmarked rows and will be ignored during window creation.
Using known Mindrove sampling rate: 500 Hz
Window size: 200 samples = 0.4 seconds
Step size: 25 samples = 0.05 seconds

Rows:
Total rows: 1476620
Labeled rows: 759376
Pause/unmarked 

### Quick Sanity Check

In [4]:
print("Y DIST")
values, counts = np.unique(y, return_counts=True)
print(dict(zip(values, counts)))

print("Y_FT DIST")
values, counts = np.unique(y_ft, return_counts=True)
print(dict(zip(values, counts)))


Y DIST
{0: 3357, 1: 3237, 2: 3346, 3: 3139, 4: 3184, 5: 3389, 6: 3513, 7: 3402}
Y_FT DIST
{0: 265, 1: 263, 2: 263, 3: 265, 4: 369, 5: 296, 6: 265, 7: 261}


### Turn Data in Tasks
- Training data
- Fine Tuning Subject

In [5]:
sampler = FOMAMLTaskSampler(
    X_train = X,  
    y_train = y,   
    X_tune  = X_ft,
    y_tune  = y_ft,
    cfg     = cfg,
    n_way   = None,
)

print(sampler)

[TaskSampler] classes      : [0, 1, 2, 3, 4, 5, 6, 7]
[TaskSampler] meta-train   | classes: 8 | total windows: 26567
[TaskSampler] meta-tune    | classes: 8 | total windows: 2247
[TaskSampler] episode      | 8-way 5-shot + 10-query per class


In [6]:
# fine_tuned_model = train_fomaml(
#     model=model,
#     sampler=sampler,
#     cfg=cfg,
#     device=device
# )

## Evaluate Model Peformance

In [7]:
class EMGWindowsDataset(Dataset):
    """
    A PyTorch Dataset for EMG windows.
    """

    def __init__(
        self,
        X: np.ndarray,
        y: np.ndarray,
        augment: bool = False,
        noise_std: float = 0.01,
        gain_jitter_std: float = 0.05
    ):
        if len(X) != len(y):
            raise ValueError("X and y must have the same number of samples")

        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

        self.augment = augment
        self.noise_std = noise_std
        self.gain_jitter_std = gain_jitter_std

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].clone()
        y = self.y[idx]

        if self.augment:
            # Add very small Gaussian noise
            x = x + torch.randn_like(x) * self.noise_std

            # Add very small per-channel amplitude jitter
            gains = 1.0 + torch.randn(x.shape[1]) * self.gain_jitter_std
            gains = gains.to(x.device)
            x = x * gains.unsqueeze(0)

        return x, y
    
eval_dir = r"fine_tune_data_eval"
data_eval = preprocess_mindrove_data(eval_dir)

X_tst = data_eval["X"]
y_tst = data_eval["y"]
y_tst = np.array([label_map[label] for label in y_tst], dtype=np.int64)

test_dataset = EMGWindowsDataset(
    X_tst,
    y_tst,
    augment=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

Number of CSV files found before exclusion: 5
Number of CSV files excluded: 0
Number of CSV files used: 5

Example file used: fine_tune_data_eval\2026-05-26T14-01-07.130875.csv

Sample rate: 500 Hz
Window size: 200 samples = 0.4 seconds
Step size: 25 samples = 0.05 seconds

raw_df shape: (115742, 14)
Detected EMG channels: ['Channel1', 'Channel2', 'Channel3', 'Channel4', 'Channel5', 'Channel6', 'Channel7', 'Channel8']
Detected number of channels: 8

Label distribution including NaN pauses:
label
Extend     8012
Fist       7356
Flex       7202
Pro        7476
Radial     7446
Rest       7332
Sup        7558
Ulnar      7408
NaN       55952
Name: count, dtype: int64
Using new Mindrove row-level labels from the last CSV column.
NaN labels are pause/unmarked rows and will be ignored during window creation.
Using known Mindrove sampling rate: 500 Hz
Window size: 200 samples = 0.4 seconds
Step size: 25 samples = 0.05 seconds

Rows:
Total rows: 115742
Labeled rows: 59790
Pause/unmarked rows: 55

In [8]:
from sklearn.metrics import classification_report
import torch.nn as nn

criterion = nn.CrossEntropyLoss(
    label_smoothing=0.0
)

test_metrics = evaluate(model, test_loader, criterion, device)

print(
    f"Test Loss={test_metrics['loss']:.4f}, "
    f"Test Acc={test_metrics['accuracy']:.4f}, "
    f"Test MacroF1={test_metrics['macro_f1']:.4f}"
)
# y_true, y_pred = evaluate_full(model, test_loader, device)
# __, y_pred_ft = evaluate_full(fine_tuned_model, test_loader, device)

# print(y_true)

# target_names = vars(model_cfg.label_map)
# target_names = list(target_names.keys())

# print(classification_report(
#     y_true,
#     y_pred,
#     labels=list(range(len(target_names))),
#     target_names=target_names,
#     digits=4,
#     zero_division=0
# ))

# print(classification_report(
#     y_true,
#     y_pred_ft,
#     labels=list(range(len(target_names))),
#     target_names=target_names,    
#     digits=4,
#     zero_division=0
# ))


Test Loss=2.0817, Test Acc=0.1222, Test MacroF1=0.0343
